In [472]:
import pandas as pd
from collections import defaultdict
import json
import krippendorff
import numpy as np
from scipy import stats
import seaborn as sns
import string
import re
import collections
import statsmodels.api as sm
from statsmodels.formula.api import ols, mixedlm
import random
import matplotlib.pyplot as plt
import matplotlib.colors
#import plotly.express as px
#import geopandas
import matplotlib.pyplot as plt
import matplotlib.colors
import matplotlib.cm as cm
import os
import yaml
import crowdkit

ModuleNotFoundError: No module named 'crowdkit'

In [2]:
def results_summary_to_dataframe(results):
    '''take the result of an statsmodel results table and transforms it into a dataframe'''
    pvals = results.pvalues
    coeff = results.params
    se = results.bse
    conf_lower = results.conf_int()[0]
    conf_higher = results.conf_int()[1]

    results_df = pd.DataFrame({"p value":pvals,
                               "coeff":coeff,
                               "SE":se,
                               "conf_lower":conf_lower,
                               "conf_higher":conf_higher
                                })

    #Reordering...
    results_df = results_df.reset_index()
    results_df['term'] = results_df['index']
    return results_df

def add_star(p):
    sig = ' '
    if p < 0.05:
        sig = '*'
    else:
        sig = ' '
    if p < 0.01:
        sig = '**'
    if p < 0.001:
        sig = '***'
   
    return sig

        
def plot_reg_results(df, nrows = 1, ncols = 1, figsize=(4, 2), xlabel='xlabel', labels = None, title='', filename=None):
    #df = df.sort_values('coeff')
    xs = df['coeff']
    norm = matplotlib.colors.Normalize(vmin=min(xs)*2, vmax=max(xs)*2)
    mapper = cm.ScalarMappable(norm=norm, cmap='viridis')
    colors = np.array([(mapper.to_rgba(v)) for v in xs])
    en_len = int(len(df))
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize)
    axes=[axes] if nrows + ncols <= 2 else axes
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.6)
    #plt.title('Uncertainty in Fields')
    st = fig.suptitle("", fontsize=14)
    #sigg = fig.suptitle("Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1", fontsize=14)
    for i in range(len(axes)):
        ax = axes[i]
        ax.set_ylim(-0.7, len(df['term'][:en_len])-0.2)
        ax.axvline(x=0.00,color='black',linewidth=1.7,linestyle='-')
        ax.xaxis.grid(True)
        ax.yaxis.label.set_color('white')
        #ax.set_xticks([-0.5, 0.0, 0.5, 1,0])
        #ax.set_xlabel(r"%s ($\beta$ coef.)"%xlabel, fontsize=14)
        ax.set_xlabel(xlabel,fontsize=14)
        #ax.set_xticks([-0.04, -0.02, 0, 0.02, 0.04])
        #ax.set_title(r"%s"%(labels[i] if labels else dvs[i]), fontsize=14)
        s = en_len*i
        en = en_len*(i+1)
        for x, y, e, p, color in zip(df['coeff'][s:en], df['term'][s:en], df['SE'][s:en],df['p value'][s:en], colors[s:en]):
            #plt.plot(x, y, 'o', color=color)
            ax.errorbar(x=[x], y=[y], xerr=[e], color=color,fmt='o')
            sig = ' '
            if p < 0.05:
                sig = '*'
            if p < 0.01:
                sig = '**'
            if p < 0.001:
                sig = '***'
            ax.annotate(sig, # this is the text
                     (x+0.00001,y), # this is the point to label
                     textcoords="offset points", # how to position the text
                     xytext=(0,-0.5), # distance from text to points (x,y)
                     ha='center') # horizontal alignment can be left, right or center

        #ax.set_xticklabels([-0.04, -0.02, 0, 0.02, 0.04], fontsize=14)
        ax.set_yticklabels(list(df['term']), fontsize=14)

    #fig.text(0.5, 0.04, 'common X', ha='center')
    st.set_y(-0.02)
    #sigg.set_y(-0.12)
    fig.text(0.5, -0.17, s="%s"%title,ha='center',fontsize=14)
    fig.text(0.5, -0.18, s="Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05",ha='center',fontsize=9)
    if filename:
        plt.savefig('../figures/%s.pdf'%filename, bbox_inches='tight')

In [447]:
def load_schema_dict(path):
    config = {}
    with open(path, "r") as file_p:
        config.update(yaml.safe_load(file_p))
    
    schema_dict = {}
    for it in config['annotation_schemes']:
        #if it['annotation_type'] in ['radio', 'multiselect']:
        #    it['label2value'] = {(l if type(l) == str else l['name']):str(i+1) for i,l in enumerate(it['labels'])}
        schema_dict[it['name']] = it
    return schema_dict



schema_dict = load_schema_dict('../configs/email_understanding.yaml')
value2label = {"[Q] Response Not-Expected":0, "[W] Response Expected":1, '[Q] Ignore':0, '[W] Accountable non-answer':0
               , "[E] Postponed reply":1, "[R] Immediate reply":1,'Other': None,
               'Spam or empty message': None}
def convert_data(df, multirate_keys = None, question_dict=None):
    return_dict = defaultdict(list)
    cols = list(df.columns)
    for i, row in df.iterrows():
        item = {}
        for key_label in cols:
            key = key_label.split(":::")
            if len(key) == 1:
                item[key[0]] = row[key_label]
            elif key[0] == 'span_annotation':
                item[key[0]] = eval(row[key_label])
            elif key[1]=='bad_text':
                item['bad_text'] = row[key_label]
            elif len(key) == 2:
                #print(key_label, row[key_label])
                if multirate_keys and key[0] in multirate_keys:
                    if str(row[key_label]) == 'nan':
                        item[key_label] = np.nan
                    else:
                        item[key_label] = label2score[row[key_label]]
                        
                elif schema_dict[key[0]]['annotation_type'] == 'multiselect':
                    if str(row[key_label]) == 'nan':
                        item[key_label] = 0
                    else:
                        item[key_label] = 1
                    
                    new_key = key[0] + '_set'
                    if new_key not in item:
                        item[new_key] = set()
                    if str(row[key_label]) != 'nan':
                        item[new_key].add(key[1].split('] ')[-1])
                elif schema_dict[key[0]]['annotation_type'] == 'radio':
                    #print(value2label[row[key_label]])
                    if key[0] in item and item[key[0]] != None:
                        continue
                    if str(row[key_label]) == 'nan':
                        continue
                        
                    if 'spam' not in item:
                        item['spam'] = 0
                    
                    if key[1] in ['Spam or empty message', 'Other']:
                        item['spam'] = 1
                    
                    item[key[0]] = value2label[key[1]] if key[1] in value2label else None
                
                elif question_dict and key[0] in question_dict:
                    if question_dict[key[0]]['schema'] == 'radio':
                        if key[0] not in item:
                            item[key[0]] = np.nan
                        if str(row[key_label]) != 'nan':
                            item[key[0]] = key[1]
                    elif question_dict[key[0]]['schema'] == 'multiselect':
                        if key[0] not in item:
                            item[key[0]] = []
                        if str(row[key_label]) != 'nan':
                            item[key[0]].append(key[1])
                    
                else:
                    #print(str(row[key_label]))
                    if key[0] not in item:
                        item[key[0]] = None
                    
                    elif str(row[key_label]) != 'nan':
                        item[key[0]] = row[key_label]
                
                #if key[0] not in item:
                #    item[key[0]] = None
                #if row[key_label] == True:
                #    if key[1][:5] == 'scale':
                #        item[key[0]] = int(key[1].split('_')[1])
                #    else:
                #        item[key[0]] = key[1]
            else:
                print('error key: ', key)
        for key in item:
            return_dict[key].append(item[key])
    for key in return_dict:
        print(key,len(return_dict))
    return pd.DataFrame(return_dict)


def get_user_dict(ann_df):
    user_keys = set(ann_df['user'])
    user_dict = defaultdict(dict)
    for key in info_dict:
        df = ann_df[ann_df.instance_id.str.contains(key)]
        for i, row in df.iterrows():
            for k in info_dict[key]:
                user_dict[row['user']][k] = row[k]#.split(':::')[-1] if type(v)==str else v
    return user_dict

def attach_user_info(ann_df, user_dict, keys):
    for key in keys:
        ann_df[key] = [user_dict[user][key] if key in user_dict[user] else None for user in ann_df['user']]
    return ann_df

def edit_distance(str1, str2):
    # Create a table to store the edit distances for substrings
    m = len(str1)
    n = len(str2)
    dp = [[0 for _ in range(n + 1)] for _ in range(m + 1)]

    # Initialize the table
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # Compute the edit distances for substrings
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i - 1] == str2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1]) + 1

    # The edit distance between the two strings is stored in the bottom-right cell
    return dp[m][n]
def get_mean(s):
    '''
        Return the mean of a list of labels
        Input: a list of labels (can contain 'bad_text' or 'nan')
        Output: a float or None
    '''
    s = [int(it) for it in s if it!='bad_text' and str(it) != 'nan']
    return sum(s)/len(s) if len(s) > 0 else None

def split_half_corr(df, key='pure_scores', seeds = [0,1,2,3,4]):
    '''
        randomly split the labels into two groups and calculate the Pearson'r r between the two groups
        Input:
            df: the input annotation dataframe
            key: the column key to be analyzed (default to pure_scores)
        Return:
            a list of the split half correlation scores 
    '''
    p = []
    for seed in seeds:
        random.seed(seed)
        scores = [random.sample(it,len(it)) for it in df[key] if len(it) >= 4]
        sp1 = [get_mean(it[:int(len(it)/2)]) for it in scores]
        sp2 = [get_mean(it[int(len(it)/2):]) for it in scores]
        p.append(stats.pearsonr(sp1,sp2)[0])
        #print(p)
    return sum(p)/len(p)#, np.std(p)
        
def get_exp_df(ann_df):
    exp_dict = {}
    df = ann_df[ann_df.instance_id.str.contains('experience.html')]
    for i, row in df.iterrows():
        exp_dict[row['user']] = {question2key[k] if k in question2key else k:v.split(':::')[-1] if type(v)==str else v for k,v in dict(row).items()}
    exp_df = pd.DataFrame.from_records(list(exp_dict.values()))
    return exp_df

def read_csv(path, cat, batch):
    df = pd.read_csv(path,sep='\t')
    df['sample'] = cat
    df['batch'] = batch
    df['instance_id'] = [b+'_'+ str(i_id) for b, i_id in zip(df['batch'], df['instance_id'])]
    return df
def split_labels(length,ratio=[0.8,0.1,0.1],seed=0):
    val_len = int(length * (ratio[0]+ratio[1])) - int(length * ratio[0])
    test_len = length - val_len - int(length*ratio[0])
    split_labels = ['train']*int(length*ratio[0]) + ['val']*val_len + ['test']*test_len
    random.seed(seed)
    random.shuffle(split_labels)
    return split_labels

def prepare_data4model(agg_df, key = 'label'):
    statements = list(set(agg_df.statement))#[:1]
    t = agg_df.pivot(index='instance_id', columns='statement', values=label)
    #print(t)
    res = agg_df.drop_duplicates('instance_id')
    for s in statements:
        res[s] = list(t[s])
    #print(res[statements])
    #quit()
    res['labels'] = [[float(i) for i in it] for it in np.array(res[statements])]
    print(res['split'].value_counts())
    
    #print(res['labels'])
    return res, statements


In [448]:
len(set(data_df.user))

13

In [449]:
def read_csvs(files, batches):
    raw_df = pd.DataFrame()
    for f, batch in zip(raw_files, batches):
        d = pd.read_csv(f)
        d['batch'] = str(batch)
        d['id'] = [b+'_'+ str(i_id) for b, i_id in zip(d['batch'], d['id'])]
        raw_df = pd.concat([raw_df, d])
    return raw_df


raw_files = [
    '../data_files/100_sample.csv',
    '../data_files/1900_sample.csv'
]
raw_df = read_csvs(raw_files, [1,2])
    
raw_df['split'] = split_labels(len(raw_df), ratio=[0.7,0.1,0.2])
data_dict = {str(row['id']):dict(row) for i,row in raw_df.iterrows()}

In [450]:
data_df = pd.concat([
    read_csv('../annotation_output/pilot/annotated_instances.tsv','pilot','1'),
    read_csv('../annotation_output/full/annotated_instances.tsv','full','2'),
])
#print(set(data_df['Please feel free to leave any comments about our study (optional):::text_box']))

In [451]:
data_df = convert_data(data_df)
#user_dict = get_user_dict(data_df)


user 19
instance_id 19
displayed_text 19
Email Act/Intent:::[9] Thank you/Welcome 19
Email Act/Intent_set 19
Email Act/Intent:::[3] Commit/Agree 19
Email Act/Intent:::[1] Request 19
Email Act/Intent:::[4] Deliver/Informative 19
Email Act/Intent:::[5] Amend 19
Email Act/Intent:::free_response 19
Email Act/Intent:::Other 19
Email Act/Intent:::[8] Remind 19
Email Act/Intent:::[6] Refuse 19
Email Act/Intent:::[7] Introduction 19
Email Act/Intent:::[2] Propose 19
spam 19
Sender Expectation 19
sample 19
batch 19


In [452]:
s2s = {
    'Email Act/Intent:::[9] Thank you/Welcome': 'Act:::Thank you/Welcome',
    'Email Act/Intent:::[3] Commit/Agree': 'Act:::Commit/Agree',
    'Email Act/Intent:::[1] Request': 'Act:::Request',
    'Email Act/Intent:::[4] Deliver/Informative': 'Act:::Deliver/Informative',
    'Email Act/Intent:::[5] Amend': 'Act:::Amend',
    'Email Act/Intent:::free_response': 'Act:::free_response',
    'Email Act/Intent:::Other': 'Act:::Other',
    'Email Act/Intent:::[8] Remind': 'Act:::Remind',
    'Email Act/Intent:::[6] Refuse': 'Act:::Refuse',
    'Email Act/Intent:::[7] Introduction': 'Act:::Introduction',
    'Email Act/Intent:::[2] Propose': 'Act:::Propose'
}
data_df = data_df.rename(columns=s2s)

In [453]:
ann_df = data_df[(~data_df.instance_id.str.contains('html'))]
#ann_df = ann_df.rename(columns=question2key)
#ann_df = attach_user_info(ann_df, user_dict, keys=question_dict.keys())

In [454]:
len(set(ann_df.instance_id))

1013

In [455]:
ann_df['Sender Expectation'].value_counts()

Sender Expectation
0.0    699
1.0    511
Name: count, dtype: int64

In [456]:
ann_df['spam'].value_counts()

spam
0    1210
1      33
Name: count, dtype: int64

In [457]:
ann_df['intent_label_cnt'] = [len(it) for it in ann_df['Email Act/Intent_set']]

In [458]:
ann_df['Email Act/Intent_set'].value_counts()

Email Act/Intent_set
{Deliver/Informative}                                           610
{Request}                                                       264
{Deliver/Informative, Request}                                  144
{Deliver/Informative, Commit/Agree}                              31
{Other, free_response}                                           29
{Deliver/Informative, Thank you/Welcome}                         19
{Deliver/Informative, Propose}                                   16
{Thank you/Welcome}                                              15
{Other}                                                          13
{Commit/Agree}                                                   12
{Request, Thank you/Welcome}                                     10
{Propose}                                                         8
{Request, Propose}                                                7
{Deliver/Informative, Request, Propose}                           7
{Deliver/Informative, Reque

In [459]:
ann_df['intent_label_cnt'].value_counts()

intent_label_cnt
1    928
2    280
3     31
4      3
5      1
Name: count, dtype: int64

In [460]:
#sel_key = 'How frequently do you read science news?'
def get_agree_df(ann_df, column):
    a_df =  ann_df.pivot(index='instance_id', columns='user', values=column).sort_values('instance_id').reset_index()#[:-2]
    user_keys = list(set(ann_df['user']))
    p_dict = defaultdict(list)
    for i, row in a_df.iterrows():
        #print('Male',[row[it] for it in user_keys if str(row[it]) != 'nan' and user_dict[it]['gender']=='Male'])
        #print('Female',[row[it] for it in user_keys if str(row[it]) != 'nan' and user_dict[it]['gender']=='Female'])
        #male_s = 
        p_dict['scores'].append([it for it in row[user_keys] if str(it) != 'nan'])
        p_dict['users'].append([it for it in user_keys if str(row[it]) != 'nan'])
    for key in p_dict:
        a_df[key] = p_dict[key]
    #agree_df['r1'] = [it[0] for it in agree_df['scores']]
    #agree_df['r2'] = [it[1] if len(it) == 2 else '' for it in agree_df['scores']]

    #agree_df['scores'] = p_dict['scores']
    #for key in ['gender']:
    #    agree_df[key] = [[user_dict[it][key] for it in s] for s in p_dict['users']]
    a_df['scores_num'] = [len(s) for s in p_dict['scores']]
    #agree_df['scores_std'] = [np.std(s) for s in p_dict['scores']]
    #a_df['label'] = [sum(s)/len(s) if len(s) > 0 else None for s in p_dict['scores']]
    a_df['label'] = [max(s) if len(s) > 0 else None for s in p_dict['scores']]
    a_df.index = a_df.index.astype(str)
    return a_df
    

In [461]:
data_df.columns

Index(['user', 'instance_id', 'displayed_text', 'Act:::Thank you/Welcome',
       'Email Act/Intent_set', 'Act:::Commit/Agree', 'Act:::Request',
       'Act:::Deliver/Informative', 'Act:::Amend', 'Act:::free_response',
       'Act:::Other', 'Act:::Remind', 'Act:::Refuse', 'Act:::Introduction',
       'Act:::Propose', 'spam', 'Sender Expectation', 'sample', 'batch'],
      dtype='object')

In [462]:
sel_cols = [
        'Act:::Thank you/Welcome',
       'Act:::Commit/Agree', 'Act:::Request',
       'Act:::Deliver/Informative', 'Act:::Amend',
       'Act:::Remind', 'Act:::Refuse', 'Act:::Introduction',
       'Act:::Propose', 'spam', 'Sender Expectation'
 ]

In [463]:
import warnings

warnings.filterwarnings('ignore')

In [464]:
#sel_cols = [it for it in ann_df.columns if it.split(':::')[0] in ['Readability and newsworthiness', 'Practical implications and controversy']]
agree_df = None
user_keys = set(ann_df.user)
bad_users = []
sel_users = [it for it in user_keys if it not in bad_users]
t_df = ann_df[~ann_df['user'].isin(bad_users)]

agg_df = pd.DataFrame()
full_df = pd.DataFrame()
agree_res = defaultdict(list)
for i,col in enumerate(sel_cols):
    c_df = get_agree_df(t_df, col)
    c_df['statement'] = col
    #print(col)
    #c_df = c_df[c_df['scores_num']>=2]
    full_df = pd.concat([full_df, c_df])
    #print(col.split(':::')[1], '#',krippendorff.alpha(np.array(c_df[sel_users]).transpose(), level_of_measurement='ordinal'), split_half_corr(c_df, key='scores'))
    agree_res['type'].append(col.split(':::')[0])
    agree_res['statement'].append(col.split(':::')[-1])
    agree_res['alpha'].append(krippendorff.alpha(np.array(c_df[sel_users]).transpose(), level_of_measurement='nominal'))
    #agree_res['shr'].append(split_half_corr(c_df, key='scores'))
    print(col, '#',krippendorff.alpha(np.array(c_df[sel_users]).transpose(), level_of_measurement='nominal'))
    

    c_df = c_df.drop(columns=sel_users)
    c_df['dimension'] = col
    agg_df = pd.concat([agg_df, c_df])
    
    a_df = c_df
    a_df.columns = [it + '_'+col if it not in ['instance_id', 'user'] else it for it in list(a_df.columns)]
    if i==0:
        agree_df = a_df
        #agree_df.columns = [it + '_'+col if it not in ['instance_id'] else it for it in list(c_df.columns)]
    else:
        for key in a_df.columns:
            agree_df[key] = a_df[key]
    
agree_res = pd.DataFrame(agree_res)        

sel_keys = ['root_id', 'group', 'message_depth', 'message_id', 'from', 'to', 'date', 'subject', 'message_body', 'nchar_fulltext', 'nchar_body', 'nchar_log', 'nchar_code', 
            'nchar_quot', 'group1', 'group2', 'group3', 'group4', 'stratum', 'from_trees', 'id', 'cleaned_text', 'batch', 'split']
for key in sel_keys:
    agree_df[key] = [data_dict[it][key] for it in agree_df['instance_id']]
#agree_df['Mention title'] = [data_dict[it]['Mention title'] for it in agree_df['instance_id']]
#agree_df['category'] = [data_dict[it]['category'] for it in agree_df['instance_id']]
'''
for key in ['doi', 'url']:
    agg_df[key] = [data_dict[it][key] for it in agg_df['instance_id']]

agg_df['Mention title'] = [data_dict[it]['Mention title'] for it in agg_df['instance_id']]
agg_df['category'] = [data_dict[it]['category'] for it in agg_df['instance_id']]
agg_df['outlet'] = [data_dict[it]['outlet'] for it in agg_df['instance_id']]
agg_df['coverage_type'] = [data_dict[it]['coverage_type'] for it in agg_df['instance_id']]
agg_df['statement'] = [it.split(':::')[1] for it in agg_df['dimension']]
agg_df['FleschReadingEase'] = [data_dict[it]['FleschReadingEase'] for it in agg_df['instance_id']]
agg_df['big_field'] = [data_dict[it]['big_field'] for it in agg_df['instance_id']]
    
'''


Act:::Thank you/Welcome # 0.35919090814305077
Act:::Commit/Agree # 0.012126515814476768
Act:::Request # 0.6875771847200418
Act:::Deliver/Informative # 0.4431375262344611
Act:::Amend # -0.011520737327189057
Act:::Remind # 0.49655963302752293
Act:::Refuse # -0.0022831050228311334
Act:::Introduction # -0.006880733944954143
Act:::Propose # 0.09297520661157022
spam # 0.3488027124390761
Sender Expectation # 0.6508965702276082


"\nfor key in ['doi', 'url']:\n    agg_df[key] = [data_dict[it][key] for it in agg_df['instance_id']]\n\nagg_df['Mention title'] = [data_dict[it]['Mention title'] for it in agg_df['instance_id']]\nagg_df['category'] = [data_dict[it]['category'] for it in agg_df['instance_id']]\nagg_df['outlet'] = [data_dict[it]['outlet'] for it in agg_df['instance_id']]\nagg_df['coverage_type'] = [data_dict[it]['coverage_type'] for it in agg_df['instance_id']]\nagg_df['statement'] = [it.split(':::')[1] for it in agg_df['dimension']]\nagg_df['FleschReadingEase'] = [data_dict[it]['FleschReadingEase'] for it in agg_df['instance_id']]\nagg_df['big_field'] = [data_dict[it]['big_field'] for it in agg_df['instance_id']]\n    \n"

In [465]:
it = list(agree_res[(agree_res['type']!='Sender Expectation')&(~agree_res['statement'].isin(['free_response','Other']))].alpha)
sum(it)/len(it)

0.24196851106952252

In [466]:
agree_df.to_csv('../annotation_output/full/agree_df.csv')

In [467]:
col_to_drop = [it for it in agree_df.columns if it.split('_')[0] in ['users', 'dimension', 'statement', 'scores']]
cols = ['group', 'message_depth', 'from', 'to', 'date',
       'subject', 'message_body', 'nchar_fulltext', 'nchar_body', 'nchar_log',
       'nchar_code', 'nchar_quot', 'group1', 'group2', 'group3', 'group4',
       'stratum', 'from_trees', 'id']
df4model = agree_df.drop(columns=col_to_drop+cols)

In [468]:
df4model.to_csv('../annotation_output/full/df4model.csv')

In [471]:
df4model['label_Sender Expectation'].value_counts()

label_Sender Expectation
0.0    572
1.0    417
Name: count, dtype: int64

In [417]:
#all_df.drop_duplicates('user')['sample'].value_counts()

In [36]:
agg_df['single_field'] = [data_dict[it]['single_field'] for it in agg_df['instance_id']]


In [37]:
agg_df['cleaned_text'] = [data_dict[it]['cleaned_text'] for it in agg_df['instance_id']]
agg_df['statement_full'] = agg_df['statement']
agg_df['statement'] = [s2s[it] for it in agg_df['statement_full']]

In [38]:
agg_df['split'] = [data_dict[it]['split'] for it in agg_df['instance_id']]

In [39]:
len(set(agg_df.cleaned_text))

1506

In [40]:
agg_df = agg_df[~agg_df.label.isna()]

In [41]:
#agg_df.drop(columns=['cleaned_text']).to_csv('annotation_output/full/batch_1/agg_df.csv',index=False)
agg_df.to_csv('annotation_output/full/agg_df_with_full_text.csv',index=False)

In [69]:
agg_df['label']

0       1.571429
1       3.333333
2       3.000000
3       1.571429
4       2.875000
          ...   
1499    1.000000
1500    1.333333
1501    2.333333
1502    1.333333
1503    0.666667
Name: label, Length: 37600, dtype: float64

In [81]:
df4model[['instance_id','cleaned_text','labels','split']].to_csv('merged.csv')

In [538]:
agree_res.to_csv('annotation_output/full/batch_1/agree_res.csv',index=False)